## Aplicação da extrategia de investimento do Basin



### Criterios:
- Payout de no máximo 50% nos últimos 5 anos
- Ter dívida/PL < 1,75>
- Ter um dividend yeld superior a 6%

In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import investpy as inv
from datetime import timedelta

In [44]:
bolsa_br = inv.stocks.get_stocks(country= 'Brazil')
bolsa_br

,country,name,full_name,isin,currency,symbol
0,brazil,ABC BRASIL PN,Banco ABC Brasil SA,BRABCBACNPR4,BRL,ABCB4
1,brazil,BRASILAGRO ON,BrasilAgro - Co ON NM,BRAGROACNOR7,BRL,AGRO3
2,brazil,RUMO ON NM,RUMO Logistica Operadora Multimodal SA,BRRAILACNOR9,BRL,RAIL3
3,brazil,ALPARGATAS ON,Alpargatas SA,BRALPAACNOR0,BRL,ALPA3
4,brazil,ALPARGATAS PN,Alpargatas SA,BRALPAACNPR7,BRL,ALPA4
...,...,...,...,...,...,...
744,brazil,Integral Brei Reit,Fdo Inv Imob Fof Integral Brei Reit,BRIBFFCTF007,BRL,IBFF11
745,brazil,Vbi Cri,Fi Imobiliario Vbi Cri,BRCVBICTF001,BRL,CVBI11
746,brazil,Hedge Realty,Hedge Realty Devl Fdo Inv Imob Etf,BRHRDFCTF000,BRL,HRDF11
747,brazil,Rb Cap,Rb Cap Desenvolvimento Res Iii Fii,BRRSPDCTF006,BRL,RSPD11


In [45]:
ticket = bolsa_br["symbol"]
ticket

0       ABCB4
1       AGRO3
2       RAIL3
3       ALPA3
4       ALPA4
        ...  
744    IBFF11
745    CVBI11
746    HRDF11
747    RSPD11
748    TCPF11
Name: symbol, Length: 749, dtype: object

In [46]:
# Filtro para pegar somente ações e excluir os fundos
lista_acoes = [acao+'.SA' for acao in ticket if len(acao) == 5]
len((lista_acoes))

390

In [47]:
# Tempo que será calculo para definir as médias de payout, divida e dividend yeld
PERIODO = 5
ano_atual = '2024-12-30'
ano_atual = pd.to_datetime(ano_atual)

ano_inicio = ano_atual - timedelta(days=365*PERIODO)



In [48]:
# Realizar filto baseado no setor, devido ao filtro usar divida como indicador, empresas do setor financeiros são excluidas do método
lista_acoes_sem_setor_financeiro = []
for acao in lista_acoes:
    empresa = yf.Ticker(acao)
    try:
        informacoes = empresa.info
        if informacoes:
            setor = informacoes.get("sector")
            if setor != "Financial Services":
                lista_acoes_sem_setor_financeiro.append(acao)
    except:
        continue

404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/ALSO3.SA?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=ALSO3.SA&crumb=.gJOwptfhXD
404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/ARZZ3.SA?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=ARZZ3.SA&crumb=.gJOwptfhXD
404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/APER3.SA?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=APER3.SA&crumb=.gJOwptfhXD
404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/CIEL3.SA?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDe

In [49]:
len(lista_acoes_sem_setor_financeiro)

292

In [50]:
# Definir o preco médio neste périodo
df_acoes = pd.DataFrame(lista_acoes_sem_setor_financeiro,columns= ['ticker'])

In [51]:
contador = 0
while contador < len(df_acoes):
    empresa = yf.Ticker(df_acoes.loc[contador,'ticker'])
    precos = empresa.history(start = ano_inicio, end= ano_atual, period= '1y')
    preco_medio = precos['Close'].mean()
    df_acoes.loc[contador, "Preco_Medio"] = preco_medio
    contador += 1



$BBRK3.SA: possibly delisted; no timezone found
$BRML3.SA: possibly delisted; no timezone found
$BTOW3.SA: possibly delisted; no timezone found
$CAMB4.SA: possibly delisted; no timezone found
$CARD3.SA: possibly delisted; no timezone found
$CCPR3.SA: possibly delisted; no timezone found
$CESP6.SA: possibly delisted; no timezone found
$CRDE3.SA: possibly delisted; no timezone found
$DTEX3.SA: possibly delisted; no timezone found
$TIET3.SA: possibly delisted; no timezone found
$TIET4.SA: possibly delisted; no timezone found
$HGTX3.SA: possibly delisted; no timezone found
$IGTA3.SA: possibly delisted; no timezone found
$LAME3.SA: possibly delisted; no timezone found
$LAME4.SA: possibly delisted; no timezone found
$LLIS3.SA: possibly delisted; no timezone found
$MMXM3.SA: possibly delisted; no timezone found
$TESA3.SA: possibly delisted; no timezone found
$CESP3.SA: possibly delisted; no timezone found
$LCAM3.SA: possibly delisted; no timezone found
$BSEV3.SA: possibly delisted; no timezon

In [52]:
len(df_acoes)

292

In [53]:
# Coletando lucro médio dos ultimos 5 anos de cada empresa
contador = 0
while contador < len(df_acoes):
    empresa = yf.Ticker(df_acoes.loc[contador,'ticker'])
    dados_financeiros = empresa.financials
    informacoes = empresa.info
    acoes = informacoes.get('sharesOutstanding')
    df_acoes.loc[contador,'Acoes_negociadas'] = acoes
    if 'Net Income' in dados_financeiros.index:
        lucro = dados_financeiros.loc['Net Income']
        lucro_medio = lucro.mean()
        df_acoes.loc[contador,'Lucro_Medio'] = lucro_medio
    else:
        lucro_medio = 0
        df_acoes.loc[contador,'Lucro_Medio'] = lucro_medio
    
    contador += 1
    print(contador)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277


In [54]:
# Colocando 0 onde n se encontrou dados
df_acoes.fillna(0,inplace=True)

In [55]:
df_acoes_sem_nulo=df_acoes[df_acoes['Preco_Medio']> 0].reset_index(drop=True)
df_acoes_sem_qtd_stock_zerado = df_acoes_sem_nulo[df_acoes_sem_nulo['Acoes_negociadas']>0].reset_index(drop=True)
df_acoes_sem_lucro_negativo = df_acoes_sem_qtd_stock_zerado[df_acoes_sem_qtd_stock_zerado['Lucro_Medio']>0].reset_index(drop=True)



In [56]:
len(df_acoes_sem_lucro_negativo)

196

In [57]:
# Calculando o LUCRO POR ACAO
df_acoes_sem_lucro_negativo["Lucro_por_Acao"]= df_acoes_sem_lucro_negativo['Lucro_Medio'] / df_acoes_sem_lucro_negativo['Acoes_negociadas']

In [58]:
contador = 0
while contador < len(df_acoes_sem_lucro_negativo):
    for acao in df_acoes_sem_lucro_negativo["ticker"]:
        empresa=yf.Ticker(acao)
        dividendo = empresa.dividends
        if not dividendo.empty:
            dividendos_filtrados = dividendo.loc[dividendo.index.year>= (ano_atual.year - PERIODO)]
            dividendos_anuais_filtrado = dividendos_filtrados.groupby(dividendos_filtrados.index.year).sum()
            if dividendos_anuais_filtrado.empty:
                dividendos_media = 0 
            else:
                dividendos_media = dividendos_anuais_filtrado.mean()
        df_acoes_sem_lucro_negativo.loc[contador, "Dividendo_por_acao"]= dividendos_media
        contador+=1

In [59]:
df_acoes_sem_lucro_negativo['Payout']= df_acoes_sem_lucro_negativo["Dividendo_por_acao"]/df_acoes_sem_lucro_negativo['Lucro_por_Acao']*100

In [60]:
df_acoes_sem_payout_maior_50 = df_acoes_sem_lucro_negativo[df_acoes_sem_lucro_negativo["Payout"]<=50].reset_index(drop=True)

In [61]:
len(df_acoes_sem_payout_maior_50)

148

In [62]:
#Patrimonio liquido = valor patrimonial*quantidade de ações
#divida liquida = divida total - caixa
#indicador é divida liquida/patrimonio liquido
contador = 0
while contador <len(df_acoes_sem_payout_maior_50):
    for acao in df_acoes_sem_payout_maior_50["ticker"]:
        empresa= yf.Ticker(acao)
        informacoes = empresa.info
        valor_patrimonial = informacoes.get("bookValue")
        qtd_acoes = informacoes.get("sharesOutstanding")
        patrimonio_liquido = valor_patrimonial*qtd_acoes
        df_acoes_sem_payout_maior_50.loc[contador, "Patrimonio_liquido"] = patrimonio_liquido
        contador += 1



In [63]:
contador = 0
while contador <len(df_acoes_sem_payout_maior_50):
    for acao in df_acoes_sem_payout_maior_50["ticker"]:
        empresa= yf.Ticker(acao)
        balanco = empresa.balance_sheet
        divida_total = balanco.loc["Total Debt"].mean() if "Total Debt" in balanco.index else 0
        caixa = balanco.loc['Cash And Cash Equivalents'].mean() if "Cash And Cash Equivalents" in balanco.index else 0
        divida_liquida = divida_total-caixa
        df_acoes_sem_payout_maior_50.loc[contador,"Divida_Liquida" ] = divida_liquida
        contador+=1

In [64]:
df_acoes_sem_payout_maior_50['Divida_Liquida_patrimonio_liquido']= df_acoes_sem_payout_maior_50['Divida_Liquida']/df_acoes_sem_payout_maior_50['Patrimonio_liquido']
df_acoes_sem_payout_maior_50.sort_values('Divida_Liquida_patrimonio_liquido', ascending=True).head(10)

,ticker,Preco_Medio,Acoes_negociadas,Lucro_Medio,Lucro_por_Acao,Dividendo_por_acao,Payout,Patrimonio_liquido,Divida_Liquida,Divida_Liquida_patrimonio_liquido
44,PDGR3.SA,24849.749903,13948000.0,1.160512e+08,8.320279,0.000000,0.000000,-2.594328e+07,1.976186e+09,-76.173339
106,MWET4.SA,10.038109,1372000.0,3.981925e+07,29.022777,2.001873,6.897594,-9.288440e+05,5.978150e+07,-64.361184
89,CTKA4.SA,15.206597,3326970.0,3.975200e+07,11.948410,0.000000,0.000000,-3.034529e+07,5.392208e+08,-17.769502
30,INEP4.SA,4.715534,12627200.0,2.162970e+08,17.129451,0.000000,0.000000,-4.559177e+08,9.207612e+08,-2.019578
139,HAGA3.SA,3.466637,3966670.0,5.407576e+06,1.363253,0.172661,12.665390,-1.708841e+07,2.509856e+07,-1.468747
98,HETA4.SA,7.320434,338801.0,1.068467e+07,31.536703,1.146281,3.634752,-2.332936e+08,3.245813e+08,-1.391300
88,CRPG6.SA,33.056669,6518110.0,1.413418e+08,21.684468,2.460915,11.348746,1.254084e+08,-1.518692e+08,-1.210997
119,CEBR3.SA,11.066088,35920900.0,5.157320e+08,14.357435,4.758866,33.145652,5.359039e+08,-5.209300e+08,-0.972059
29,INEP3.SA,5.059105,31978700.0,2.162970e+08,6.763783,0.000000,0.000000,-1.154623e+09,9.207612e+08,-0.797456
87,CRPG5.SA,32.934710,12342200.0,1.413418e+08,11.451909,2.749901,24.012604,2.374639e+08,-1.518692e+08,-0.639547


In [65]:
df_acoes_dl_pl = df_acoes_sem_payout_maior_50[df_acoes_sem_payout_maior_50["Divida_Liquida_patrimonio_liquido"]<=1.75].reset_index(drop=True)

In [66]:
df_acoes_dl_pl["Dividend_Yeld"] = df_acoes_dl_pl["Dividendo_por_acao"] / df_acoes_dl_pl["Preco_Medio"]*100

In [67]:
df_acoes_invest = df_acoes_dl_pl[df_acoes_dl_pl['Dividend_Yeld'] < 15].reset_index(drop=True)
df_acoes_investir = df_acoes_invest[df_acoes_invest['Dividend_Yeld'] >6].reset_index(drop=True)
df_final =df_acoes_investir.sort_values("Dividend_Yeld", ascending=False).reset_index(drop=True)

In [68]:
contador =0
while contador < len(df_final):
    for acao in df_final['ticker']:
        df_final.loc[contador,"empresa"] = acao[:4]
        contador +=1
df_final

,ticker,Preco_Medio,Acoes_negociadas,Lucro_Medio,Lucro_por_Acao,Dividendo_por_acao,Payout,Patrimonio_liquido,Divida_Liquida,Divida_Liquida_patrimonio_liquido,Dividend_Yeld,empresa
0,CMIG4.SA,6.296895,1.904080e+09,4.117750e+09,2.162593,0.875393,40.478846,1.867712e+10,1.064275e+10,0.569828,13.901973,CMIG
1,GOAU3.SA,8.053631,3.651110e+08,3.228280e+09,8.841914,0.990000,11.196671,7.283234e+09,9.574508e+09,1.314596,12.292591,GOAU
2,WHRL4.SA,4.907918,4.740850e+08,5.903322e+08,1.245203,0.592049,47.546341,7.884033e+08,-3.562172e+08,-0.451821,12.063134,WHRL
3,TAEE3.SA,9.401800,5.907140e+08,1.823422e+09,3.086811,1.130507,36.623799,1.193538e+10,6.811588e+09,0.570706,12.024373,TAEE
4,GOAU4.SA,8.572379,6.335040e+08,3.228280e+09,5.095911,0.990000,19.427341,1.263714e+10,9.574508e+09,0.757648,11.548719,GOAU
5,CMIG3.SA,8.701558,9.566020e+08,4.117750e+09,4.304559,0.949666,22.061856,9.383309e+09,1.064275e+10,1.134221,10.913743,CMIG
6,TAEE4.SA,9.667267,4.427830e+08,1.823422e+09,4.118095,1.004506,24.392492,8.946431e+09,6.811588e+09,0.761375,10.390796,TAEE
7,GGBR3.SA,13.732840,7.188640e+08,9.196738e+09,12.793432,1.318452,10.305697,2.040711e+10,1.124421e+10,0.550995,9.600726,GGBR
8,CGRA3.SA,23.555191,8.599450e+06,1.150928e+08,13.383734,2.190781,16.368983,3.851092e+08,-1.023550e+07,-0.026578,9.300630,CGRA
9,CGRA4.SA,23.595529,1.191480e+07,1.150928e+08,9.659646,2.188579,22.656926,5.335805e+08,-1.023550e+07,-0.019183,9.275396,CGRA


In [69]:
yield_minimo = 0.06
CARTEIRA_SEM_CEMIG4 = df_final[df_final['ticker']!='CMIG4.SA']
CARTEIRA_SEM_GGBR4 =  CARTEIRA_SEM_CEMIG4[CARTEIRA_SEM_CEMIG4['ticker']!='GGBR4.SA']
CARTEIRA = CARTEIRA_SEM_GGBR4.drop_duplicates('empresa', keep= "first").head(10)
CARTEIRA["Preco_teto"] = CARTEIRA['Dividendo_por_acao']/yield_minimo

In [70]:
CARTEIRA

,ticker,Preco_Medio,Acoes_negociadas,Lucro_Medio,Lucro_por_Acao,Dividendo_por_acao,Payout,Patrimonio_liquido,Divida_Liquida,Divida_Liquida_patrimonio_liquido,Dividend_Yeld,empresa,Preco_teto
1,GOAU3.SA,8.053631,3.651110e+08,3.228280e+09,8.841914,0.990000,11.196671,7.283234e+09,9.574508e+09,1.314596,12.292591,GOAU,16.500000
2,WHRL4.SA,4.907918,4.740850e+08,5.903322e+08,1.245203,0.592049,47.546341,7.884033e+08,-3.562172e+08,-0.451821,12.063134,WHRL,9.867478
3,TAEE3.SA,9.401800,5.907140e+08,1.823422e+09,3.086811,1.130507,36.623799,1.193538e+10,6.811588e+09,0.570706,12.024373,TAEE,18.841792
5,CMIG3.SA,8.701558,9.566020e+08,4.117750e+09,4.304559,0.949666,22.061856,9.383309e+09,1.064275e+10,1.134221,10.913743,CMIG,15.827761
7,GGBR3.SA,13.732840,7.188640e+08,9.196738e+09,12.793432,1.318452,10.305697,2.040711e+10,1.124421e+10,0.550995,9.600726,GGBR,21.974206
8,CGRA3.SA,23.555191,8.599450e+06,1.150928e+08,13.383734,2.190781,16.368983,3.851092e+08,-1.023550e+07,-0.026578,9.300630,CGRA,36.513019
10,CPLE6.SA,6.468295,1.671980e+09,3.056898e+09,1.828310,0.591317,32.342270,1.429041e+10,8.768804e+09,0.613614,9.141775,CPLE,9.855283
11,UNIP6.SA,51.866810,7.102270e+07,1.116465e+09,15.719829,4.656027,29.618812,1.727911e+09,6.600368e+08,0.381985,8.976890,UNIP,77.600442
13,PATI4.SA,32.855711,1.025290e+06,1.362381e+08,132.877643,2.912733,2.192042,3.888720e+07,2.757518e+07,0.709107,8.865227,PATI,48.545556
14,ROMI3.SA,10.555912,9.317070e+07,1.741925e+08,1.869606,0.919676,49.190892,1.226779e+09,1.578358e+08,0.128659,8.712425,ROMI,15.327931


In [71]:
CARTEIRA_BASIN = CARTEIRA[['ticker','Preco_teto']]

In [72]:
CARTEIRA_BASIN

,ticker,Preco_teto
1,GOAU3.SA,16.500000
2,WHRL4.SA,9.867478
3,TAEE3.SA,18.841792
5,CMIG3.SA,15.827761
7,GGBR3.SA,21.974206
8,CGRA3.SA,36.513019
10,CPLE6.SA,9.855283
11,UNIP6.SA,77.600442
13,PATI4.SA,48.545556
14,ROMI3.SA,15.327931


In [73]:
# Para definir o valor do salario a ser utilizado será utilizado uma função que irá retornar o valor em questão
def investimento_ano(ano):
    SALARIOS = {2000:151,2001:180,2002:200, 2003:240, 2004:260, 2005:300,2006:350,2007:380,2008:415,2009:465,2010:510,2011:540,2012:622,2013:678,
            2014:724,2015:788,2016:880,2017:937,2018:954,2019:998,2020:1039,2021:1100,2022:1212,2023:1302,2024:1412}
    TAXA_INVEST = 0.075
    ano_atual = {}
    for anos in SALARIOS.keys():
        if anos == ano:
            ano_atual[ano]= SALARIOS[ano]
            valor_investido = ano_atual[ano]*TAXA_INVEST
    return round(valor_investido,2)

In [74]:
def correcao(ano,preco):
    INFLACAO_MEDIA = 0.0618
    ano_atual=pd.Timestamp.now().year
    periodo_correcao = ano_atual- ano
    correcao = (1+INFLACAO_MEDIA)**periodo_correcao
    preco_teto_corrigido = round((preco/correcao),2)
    return round(preco_teto_corrigido,2)

In [75]:
# Tempo do backtest
ano_inicio = '2000-01-01'
ano_atual = '2024-12-31'

In [76]:
preco_ativos = {}
for ativos in CARTEIRA_BASIN['ticker']:
    empresa = yf.Ticker(ativos)
    dados = empresa.history(start='2000-01-01', end='2024-12-31', period='1mo')
    preco = dados['Close']
    preco_ativos[ativos] = preco.resample('M').last()
precos = pd.DataFrame(preco_ativos)



C:\Users\edens\AppData\Local\Temp\ipykernel_21976\4202783768.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  preco_ativos[ativos] = preco.resample('M').last()
C:\Users\edens\AppData\Local\Temp\ipykernel_21976\4202783768.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  preco_ativos[ativos] = preco.resample('M').last()
C:\Users\edens\AppData\Local\Temp\ipykernel_21976\4202783768.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  preco_ativos[ativos] = preco.resample('M').last()
C:\Users\edens\AppData\Local\Temp\ipykernel_21976\4202783768.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  preco_ativos[ativos] = preco.resample('M').last()
C:\Users\edens\AppData\Local\Temp\ipykernel_21976\4202783768.py:6: FutureWarning: 'M' is deprecated and will be removed in a fut

In [77]:
bolsa_basin = precos.reset_index()
bolsa_basin #Tirar ggbr4

,Date,GOAU3.SA,WHRL4.SA,TAEE3.SA,CMIG3.SA,GGBR3.SA,CGRA3.SA,CPLE6.SA,UNIP6.SA,PATI4.SA,ROMI3.SA
0,2000-01-31 00:00:00-02:00,0.921254,0.283719,NaN,0.486964,0.179640,NaN,0.474200,19.425749,NaN,0.149790
1,2000-02-29 00:00:00-03:00,0.223116,0.283719,NaN,0.442694,0.189905,NaN,0.522919,19.597649,NaN,0.148427
2,2000-03-31 00:00:00-03:00,0.223116,0.232650,NaN,0.486964,0.164242,NaN,0.479072,24.067284,NaN,0.148578
3,2000-04-30 00:00:00-03:00,0.217718,0.232650,NaN,0.453762,0.179640,NaN,0.433601,24.411140,NaN,0.166602
4,2000-05-31 00:00:00-03:00,0.737003,0.232650,NaN,0.464829,0.637166,NaN,0.477448,0.817083,NaN,0.166602
...,...,...,...,...,...,...,...,...,...,...,...
295,2024-08-31 00:00:00-03:00,10.426331,4.134343,11.544510,14.014286,15.958160,24.341160,10.037450,46.772297,37.318066,10.502937
296,2024-09-30 00:00:00-03:00,10.614102,4.457757,11.181598,14.385393,16.607908,23.631920,9.806483,46.129280,29.000000,10.566438
297,2024-10-31 00:00:00-03:00,10.050785,4.275612,11.367958,14.316328,17.582527,23.631920,9.600332,44.220005,31.500000,9.854204
298,2024-11-30 00:00:00-03:00,11.380000,4.122227,11.320000,14.799788,19.600000,23.830507,9.502470,52.000000,33.500000,9.024889


In [78]:
SALARIOS = {2000:151,2001:180,2002:200, 2003:240, 2004:260, 2005:300,2006:350,2007:380,2008:415,2009:465,2010:510,2011:540,2012:622,2013:678,
            2014:724,2015:788,2016:880,2017:937,2018:954,2019:998,2020:1039,2021:1100,2022:1212,2023:1302,2024:1412}

In [79]:
for ano in SALARIOS.keys():
    for acao in CARTEIRA_BASIN['ticker']:
        Preco_teto = CARTEIRA_BASIN.loc[CARTEIRA_BASIN['ticker']== acao,"Preco_teto"]
        CARTEIRA_BASIN.loc[CARTEIRA_BASIN['ticker']== acao,ano] = correcao(ano,Preco_teto)

C:\Users\edens\AppData\Local\Temp\ipykernel_21976\24156561.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  CARTEIRA_BASIN.loc[CARTEIRA_BASIN['ticker']== acao,ano] = correcao(ano,Preco_teto)
C:\Users\edens\AppData\Local\Temp\ipykernel_21976\24156561.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  CARTEIRA_BASIN.loc[CARTEIRA_BASIN['ticker']== acao,ano] = correcao(ano,Preco_teto)
C:\Users\edens\AppData\Local\Temp\ipykernel_21976\24156561.py:4: SettingWithCopyWarning: 
A value is trying to be set on a c

In [80]:
CARTEIRA_BASIN

,ticker,Preco_teto,2000,2001,2002,2003,2004,2005,2006,2007,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
1,GOAU3.SA,16.500000,3.68,3.91,4.15,4.41,4.68,4.97,5.28,5.61,...,9.06,9.62,10.21,10.84,11.51,12.23,12.98,13.78,14.64,15.54
2,WHRL4.SA,9.867478,2.20,2.34,2.48,2.64,2.80,2.97,3.16,3.35,...,5.42,5.75,6.11,6.48,6.89,7.31,7.76,8.24,8.75,9.29
3,TAEE3.SA,18.841792,4.21,4.47,4.74,5.04,5.35,5.68,6.03,6.40,...,10.34,10.98,11.66,12.38,13.15,13.96,14.82,15.74,16.71,17.75
5,CMIG3.SA,15.827761,3.53,3.75,3.99,4.23,4.49,4.77,5.07,5.38,...,8.69,9.23,9.80,10.40,11.04,11.73,12.45,13.22,14.04,14.91
7,GGBR3.SA,21.974206,4.91,5.21,5.53,5.87,6.24,6.62,7.03,7.47,...,12.06,12.81,13.60,14.44,15.33,16.28,17.29,18.36,19.49,20.70
8,CGRA3.SA,36.513019,8.15,8.66,9.19,9.76,10.36,11.01,11.69,12.41,...,20.05,21.28,22.60,24.00,25.48,27.05,28.73,30.50,32.39,34.39
10,CPLE6.SA,9.855283,2.20,2.34,2.48,2.63,2.80,2.97,3.15,3.35,...,5.41,5.74,6.10,6.48,6.88,7.30,7.75,8.23,8.74,9.28
11,UNIP6.SA,77.600442,17.33,18.40,19.54,20.75,22.03,23.39,24.83,26.37,...,42.60,45.24,48.03,51.00,54.15,57.50,61.05,64.82,68.83,73.08
13,PATI4.SA,48.545556,10.84,11.51,12.22,12.98,13.78,14.63,15.54,16.50,...,26.65,28.30,30.05,31.90,33.88,35.97,38.19,40.55,43.06,45.72
14,ROMI3.SA,15.327931,3.42,3.63,3.86,4.10,4.35,4.62,4.91,5.21,...,8.42,8.94,9.49,10.07,10.70,11.36,12.06,12.80,13.60,14.44


In [81]:
#Colocar zero onde tiver NAN
bolsa_basin.fillna(0,inplace=True)

In [82]:
# Comprar acoes
#mesmo contando o df no preco teto ele ainda retorna um dataframe, para ter o valor realmente precisa coverte-lo, pois ele retorna uma matriz panda series, por isso o .values[0]
contador = 0
saldo=0
carteira_contabil_basin = 0
valor_investido_basin = 0
compras_basin ={acao:0 for acao in CARTEIRA_BASIN['ticker']}
while contador < len(bolsa_basin):
    data = bolsa_basin.loc[contador,"Date"].year
    saldo+=investimento_ano(data)
    valor_investido_basin+=investimento_ano(data)
    while True:
        comprar_acao = False
        for acao in CARTEIRA_BASIN['ticker']:
            preco= bolsa_basin.loc[contador,acao]
            Preco_teto = CARTEIRA_BASIN.loc[CARTEIRA_BASIN['ticker']==acao,data].values[0]
            if preco >0 and preco < saldo and preco< Preco_teto:
                compras_basin[acao]+=1
                saldo-=preco
                carteira_contabil_basin += preco
                comprar_acao = True
        if not comprar_acao:
            break
    contador+=1
print(compras_basin)
print(carteira_contabil_basin)
print(valor_investido_basin)

{'GOAU3.SA': 399, 'WHRL4.SA': 707, 'TAEE3.SA': 359, 'CMIG3.SA': 535, 'GGBR3.SA': 445, 'CGRA3.SA': 331, 'CPLE6.SA': 525, 'UNIP6.SA': 352, 'PATI4.SA': 174, 'ROMI3.SA': 368}
14968.98564620316
14973.120000000006


In [83]:
file = pd.ExcelWriter('Arquivo_Basin.xlsx', engine='openpyxl')
bolsa_basin_salv= bolsa_basin.fillna(0)
bolsa_basin_salv['Date'] = bolsa_basin['Date'].dt.strftime('%Y-%m-%d')
bolsa_basin_salv.to_excel(file, sheet_name='Bolsa Basin', index=False)
dicionario_inf = {
    'Nome':['carteira_contabil_basin','valor_investido_basin'],
    'Valor':[carteira_contabil_basin,valor_investido_basin]}
df_valor = pd.DataFrame(dicionario_inf)
df_valor.to_excel(file, sheet_name='valor_investido_vs_investimento', index=False)
compra_feita = {
    'Acao' : [acao for acao in compras_basin.keys()],
    'Qtd_comprada': [valor for valor in compras_basin.values()]}
compra_df = pd.DataFrame(compra_feita)
compra_df.to_excel(file, sheet_name='Compras', index=False)
file.close()